In [4]:
SENTENCE = "Then to understand the details of the technology, to understand also the mechanism of standardization and using Internet technology."

In [5]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer

MODEL = "rrivera1849/LUAR-MUD"
MAXLEN = 32
DEV = "cuda" if torch.cuda.is_available() else "cpu"

label_map = pd.read_csv("label_map.csv").set_index("label")
K = len(label_map)

In [6]:
# LUAR encoder (frozen, same call signature as embed.py)
tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
encoder = AutoModel.from_pretrained(MODEL, trust_remote_code=True).to(DEV).eval()

# classifier head, same architecture as train_minimal.py: 512 -> 2048 -> 2048 -> 2048 -> K
head = nn.Sequential(
    nn.Linear(512, 2048), nn.ReLU(),
    nn.Linear(2048, 2048), nn.ReLU(),
    nn.Linear(2048, 2048), nn.ReLU(),
    nn.Linear(2048, K),
).to(DEV)
head.load_state_dict(torch.load("head.pt", map_location=DEV))
head.eval()

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Sequential(
  (0): Linear(in_features=512, out_features=2048, bias=True)
  (1): ReLU()
  (2): Linear(in_features=2048, out_features=2048, bias=True)
  (3): ReLU()
  (4): Linear(in_features=2048, out_features=2048, bias=True)
  (5): ReLU()
  (6): Linear(in_features=2048, out_features=50, bias=True)
)

In [7]:
@torch.no_grad()
def embed(sentence: str) -> torch.Tensor:
    t = tok([sentence], max_length=MAXLEN, padding="max_length", truncation=True,
            return_tensors="pt")
    ids = t.input_ids.reshape(1, 1, -1).to(DEV)
    mask = t.attention_mask.reshape(1, 1, -1).to(DEV)
    out = encoder(input_ids=ids, attention_mask=mask)
    return out.float()

@torch.no_grad()
def predict(sentence: str) -> pd.DataFrame:
    x = embed(sentence)
    probs = torch.softmax(head(x), dim=1).squeeze(0).cpu().numpy()
    df = label_map.copy()
    df["probability"] = probs
    return df.sort_values("probability", ascending=False)[["lecturer_name", "probability"]]

In [8]:
dist = predict(SENTENCE)
print(f"most likely: {dist.iloc[0].lecturer_name}  (p={dist.iloc[0].probability:.3f})\n")
dist.head(10)

most likely: Prof. Dr. Christoph Meinel  (p=1.000)



,lecturer_name,probability
label,,
0,Prof. Dr. Christoph Meinel,9.998289e-01
36,Prof. Dr. Bert Arnrich,1.710914e-04
4,Prof. Dr. Andreas Polze,7.138999e-12
19,Dipl. Inf. Stefan Neumann,1.131470e-13
1,Prof. Dr. Harald Sack,8.771618e-14
33,Prof. Dr. Erwin Böttinger,6.236700e-14
46,Prof. Dr. Ralf Herbrich,1.620402e-15
22,Dr. Daniel Oberle,1.331194e-15
44,Ilin Tolovski,5.236666e-17
